In [2]:
!pip install -q fastapi uvicorn scikit-learn joblib requests pyyaml

In [3]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Create model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
model.fit(X, y)

# Save model
joblib.dump(model, "model.pkl")

print("Model training completed successfully.")
print("Saved model as model.pkl")

Model training completed successfully.
Saved model as model.pkl


In [4]:
import joblib
import numpy as np

model = joblib.load("model.pkl")

sample = np.array([[5.1, 3.5, 1.4, 0.2]])

prediction = model.predict(sample)

print("Input:", sample.tolist())
print("Predicted class:", prediction[0])

Input: [[5.1, 3.5, 1.4, 0.2]]
Predicted class: 0


In [5]:
%%writefile main.py

from fastapi import FastAPI, Request
import joblib
import numpy as np

app = FastAPI(
    title="Iris ML Prediction API",
    description="Containerized Machine Learning API using FastAPI",
    version="1.0"
)

# Load trained model
model = joblib.load("model.pkl")


@app.get("/")
def home():
    return {
        "message": "ML Model API is running",
        "model": "Random Forest",
        "dataset": "Iris"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post("/predict")
async def predict(request: Request):
    body = await request.json()
    features = body["features"]

    data = np.array(features).reshape(1, -1)

    prediction = model.predict(data)
    probabilities = model.predict_proba(data)

    return {
        "prediction": int(prediction[0]),
        "probability": float(max(probabilities[0]))
    }

Writing main.py


In [6]:
import os

print(os.path.exists("main.py"))

True


In [7]:
import importlib
import main

importlib.reload(main)

print("FastAPI application loaded successfully.")

FastAPI application loaded successfully.


In [8]:
from fastapi.testclient import TestClient

client = TestClient(main.app)

print("FastAPI application loaded successfully.")

FastAPI application loaded successfully.


In [9]:
response = client.get("/")

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'message': 'ML Model API is running', 'model': 'Random Forest', 'dataset': 'Iris'}


In [10]:
response = client.get("/health")

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'status': 'healthy'}


In [11]:
response = client.post(
    "/predict",
    json={
        "features": [5.1, 3.5, 1.4, 0.2]
    }
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'prediction': 0, 'probability': 1.0}


In [12]:
%%writefile Dockerfile

FROM python:3.10-slim

WORKDIR /app

COPY main.py .
COPY model.pkl .

RUN pip install --no-cache-dir \
    fastapi \
    uvicorn \
    scikit-learn \
    numpy \
    joblib

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


In [13]:
%%writefile .dockerignore

__pycache__
*.pyc
.ipynb_checkpoints
*.ipynb
.git
.env

Writing .dockerignore


In [14]:
!ls -lah

total 216K
drwxr-xr-x 1 root root 4.0K Aug 26 05:43 .
drwxr-xr-x 1 root root 4.0K Aug 26 05:16 ..
drwxr-xr-x 4 root root 4.0K Aug 24 13:21 .config
-rw-r--r-- 1 root root  255 Aug 26 05:42 Dockerfile
-rw-r--r-- 1 root root   56 Aug 26 05:43 .dockerignore
-rw-r--r-- 1 root root  871 Aug 26 05:39 main.py
-rw-r--r-- 1 root root 183K Aug 26 05:37 model.pkl
drwxr-xr-x 2 root root 4.0K Aug 26 05:40 __pycache__
drwxr-xr-x 1 root root 4.0K Aug 24 13:21 sample_data


In [15]:
print("========== DOCKER CONFIGURATION ==========")

with open("Dockerfile", "r") as f:
    dockerfile = f.read()

print(dockerfile)

========== DOCKER CONFIGURATION ==========

FROM python:3.10-slim

WORKDIR /app

COPY main.py .
COPY model.pkl .

RUN pip install --no-cache-dir \
    fastapi \
    uvicorn \
    scikit-learn \
    numpy \
    joblib

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]



In [16]:
import os

required_files = [
    "Dockerfile",
    "main.py",
    "model.pkl",
    ".dockerignore"
]

print("Docker project validation")
print("=" * 40)

all_present = True

for file in required_files:
    exists = os.path.exists(file)
    print(f"{file}: {'✓ Present' if exists else '✗ Missing'}")

    if not exists:
        all_present = False

print("=" * 40)

if all_present:
    print("Docker project structure is valid.")
else:
    print("Docker project structure is incomplete.")

Docker project validation
Dockerfile: ✓ Present
main.py: ✓ Present
model.pkl: ✓ Present
.dockerignore: ✓ Present
Docker project structure is valid.


In [17]:
import json
import os

docker_manifest = {
    "image_name": "iris-ml-api",
    "base_image": "python:3.10-slim",
    "working_directory": "/app",
    "application": "FastAPI",
    "model": "RandomForestClassifier",
    "model_file": "model.pkl",
    "application_file": "main.py",
    "port": 8000,
    "command": [
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
}

with open("docker-image-manifest.json", "w") as f:
    json.dump(docker_manifest, f, indent=4)

print("Docker image manifest created.")

Docker image manifest created.


In [18]:
!cat docker-image-manifest.json

{
    "image_name": "iris-ml-api",
    "base_image": "python:3.10-slim",
    "working_directory": "/app",
    "application": "FastAPI",
    "model": "RandomForestClassifier",
    "model_file": "model.pkl",
    "application_file": "main.py",
    "port": 8000,
    "command": [
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
}

In [19]:
%%writefile deployment.yaml

apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-ml-deployment
spec:
  replicas: 2
  selector:
    matchLabels:
      app: iris-ml
  template:
    metadata:
      labels:
        app: iris-ml
    spec:
      containers:
      - name: iris-ml-container
        image: iris-ml-api:latest
        ports:
        - containerPort: 8000
        resources:
          requests:
            cpu: "100m"
            memory: "128Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"

Writing deployment.yaml


In [20]:
%%writefile service.yaml

apiVersion: v1
kind: Service
metadata:
  name: iris-ml-service
spec:
  selector:
    app: iris-ml
  ports:
  - protocol: TCP
    port: 8000
    targetPort: 8000
  type: ClusterIP

Writing service.yaml


In [21]:
print("========== DEPLOYMENT YAML ==========")
!cat deployment.yaml

print("\n========== SERVICE YAML ==========")
!cat service.yaml

========== DEPLOYMENT YAML ==========

apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-ml-deployment
spec:
  replicas: 2
  selector:
    matchLabels:
      app: iris-ml
  template:
    metadata:
      labels:
        app: iris-ml
    spec:
      containers:
      - name: iris-ml-container
        image: iris-ml-api:latest
        ports:
        - containerPort: 8000
        resources:
          requests:
            cpu: "100m"
            memory: "128Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"

========== SERVICE YAML ==========

apiVersion: v1
kind: Service
metadata:
  name: iris-ml-service
spec:
  selector:
    app: iris-ml
  ports:
  - protocol: TCP
    port: 8000
    targetPort: 8000
  type: ClusterIP


In [22]:
import yaml

with open("deployment.yaml") as f:
    deployment = yaml.safe_load(f)

with open("service.yaml") as f:
    service = yaml.safe_load(f)

print("Kubernetes configuration validation")
print("=" * 45)

print("Deployment:")
print(" API Version:", deployment["apiVersion"])
print(" Kind:", deployment["kind"])
print(" Name:", deployment["metadata"]["name"])
print(" Replicas:", deployment["spec"]["replicas"])
print(" Container:", deployment["spec"]["template"]["spec"]["containers"][0]["name"])

print("\nService:")
print(" API Version:", service["apiVersion"])
print(" Kind:", service["kind"])
print(" Name:", service["metadata"]["name"])
print(" Type:", service["spec"]["type"])

print("\nKubernetes YAML validation completed successfully.")

Kubernetes configuration validation
Deployment:
 API Version: apps/v1
 Kind: Deployment
 Name: iris-ml-deployment
 Replicas: 2
 Container: iris-ml-container

Service:
 API Version: v1
 Kind: Service
 Name: iris-ml-service
 Type: ClusterIP

Kubernetes YAML validation completed successfully.


In [23]:
deployment_report = {
    "application": "Iris ML API",
    "container_image": "iris-ml-api:latest",
    "replicas": 2,
    "container_port": 8000,
    "service_name": "iris-ml-service",
    "service_type": "ClusterIP",
    "deployment_status": "Configuration validated",
    "api_status": "FastAPI tested successfully"
}

for key, value in deployment_report.items():
    print(f"{key}: {value}")

application: Iris ML API
container_image: iris-ml-api:latest
replicas: 2
container_port: 8000
service_name: iris-ml-service
service_type: ClusterIP
deployment_status: Configuration validated
api_status: FastAPI tested successfully


In [24]:
test_samples = [
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.1, 4.7, 1.5],
    [6.3, 3.3, 6.0, 2.5]
]

print("Prediction Results")
print("=" * 50)

for sample in test_samples:
    response = client.post(
        "/predict",
        json={"features": sample}
    )

    result = response.json()

    print("Input:", sample)
    print("Prediction:", result["prediction"])
    print("Probability:", round(result["probability"], 4))
    print("-" * 50)

Prediction Results
Input: [5.1, 3.5, 1.4, 0.2]
Prediction: 0
Probability: 1.0
--------------------------------------------------
Input: [6.7, 3.1, 4.7, 1.5]
Prediction: 1
Probability: 1.0
--------------------------------------------------
Input: [6.3, 3.3, 6.0, 2.5]
Prediction: 2
Probability: 1.0
--------------------------------------------------


In [25]:
report = """
===========================================================
CONTAINERIZED MACHINE LEARNING APPLICATION EXPERIMENT
===========================================================

Application:
Iris Flower Classification API

Machine Learning Model:
Random Forest Classifier

Dataset:
Iris Dataset

API Framework:
FastAPI

Model File:
model.pkl

Docker Image:
iris-ml-api:latest

Application Port:
8000

Kubernetes Deployment:
iris-ml-deployment

Kubernetes Replicas:
2

Kubernetes Service:
iris-ml-service

Service Type:
ClusterIP

-----------------------------------------------------------
IMPLEMENTATION RESULTS
-----------------------------------------------------------

1. Machine learning model trained successfully.
2. Model serialized using Joblib.
3. FastAPI application created successfully.
4. Root API endpoint tested successfully.
5. Health endpoint tested successfully.
6. Prediction endpoint tested successfully.
7. Dockerfile created successfully.
8. Docker project structure validated successfully.
9. Kubernetes Deployment YAML created successfully.
10. Kubernetes Service YAML created successfully.
11. Kubernetes configuration validated successfully.

-----------------------------------------------------------
PREDICTION TEST
-----------------------------------------------------------

Input:
[5.1, 3.5, 1.4, 0.2]

Expected class:
0

Class:
Iris Setosa

-----------------------------------------------------------
CONCLUSION
-----------------------------------------------------------

A machine learning model was developed using the Iris dataset
and exposed through a FastAPI REST API. A Dockerfile was
created to containerize the application, and Kubernetes
Deployment and Service configurations were prepared for
orchestration.

The hosted Google Colab runtime does not permit the required
Docker container runtime operations, so actual Docker
container execution was not performed in the managed runtime.

The Docker and Kubernetes configurations were nevertheless
created and validated, while the ML API was fully tested
within the Colab environment.
===========================================================
"""

print(report)

with open("experiment_report.txt", "w") as f:
    f.write(report)

print("\nExperiment report saved as experiment_report.txt")


CONTAINERIZED MACHINE LEARNING APPLICATION EXPERIMENT

Application:
Iris Flower Classification API

Machine Learning Model:
Random Forest Classifier

Dataset:
Iris Dataset

API Framework:
FastAPI

Model File:
model.pkl

Docker Image:
iris-ml-api:latest

Application Port:
8000

Kubernetes Deployment:
iris-ml-deployment

Kubernetes Replicas:
2

Kubernetes Service:
iris-ml-service

Service Type:
ClusterIP

-----------------------------------------------------------
IMPLEMENTATION RESULTS
-----------------------------------------------------------

1. Machine learning model trained successfully.
2. Model serialized using Joblib.
3. FastAPI application created successfully.
4. Root API endpoint tested successfully.
5. Health endpoint tested successfully.
6. Prediction endpoint tested successfully.
7. Dockerfile created successfully.
8. Docker project structure validated successfully.
9. Kubernetes Deployment YAML created successfully.
10. Kubernetes Service YAML created successfully.
11. K

In [26]:
from google.colab import files

files.download("main.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
files.download("Dockerfile")
files.download("deployment.yaml")
files.download("service.yaml")
files.download("experiment_report.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>